# Leksjon 11 - Agent-til-agent (A2A) protokoll


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hva er A2A-protokollen?

Den **Agent-til-agent (A2A) protokollen** er en åpen standard som gjør det mulig for AI-agenter å kommunisere,
oppdage hverandre, og samarbeide — selv når de er bygd på forskjellige rammeverk eller hostet
av ulike tjenester.

Key concepts:

- **Oppdagelse** – Agenter publiserer et *Agentkort* som beskriver deres evner, noe som gjør det
  enkelt for andre agenter (eller orkestratorer) å finne riktig spesialist for en oppgave.
- **Meldingsutveksling** – Agenter utveksler strukturerte meldinger gjennom en felles protokoll, så en
  forespørsel fra en agent kan forstås og oppfylles av en annen uavhengig av dens interne
  implementering.
- **Oppgavens livssyklus** – A2A definerer tilstander som *innsendt*, *behandles*, *fullført*, og
  *mislyktes*, og gir orkestratoren full innsikt i hvordan en delegert oppgave utvikler seg.

I denne leksjonen simulerer vi A2A-stil samarbeid ved å koble tre spesialiserte reiseagenter
inn i en arbeidsflyt hvor hver agent bidrar med sin ekspertise og sender resultater videre til den neste.


## Opprette spesialiserte reiseagenter


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samarbeid mellom flere agenter via arbeidsflyt

Vi kobler de tre agentene inn i en sekvensiell arbeidsflyt som speiler A2A-meldingsutveksling:

1. **CurrencyExchangeAgent** mottar brukerens forespørsel og gir veiledning om valuta.
2. **ActivityPlannerAgent** mottar den berikede konteksten og legger til aktivitetsanbefalinger.
3. **TravelManagerAgent** syntetiserer begge innspillene til en endelig reiseoversikt.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Forstå A2A i produksjon

I et produksjonsmiljø åpner A2A-protokollen for kraftige scenarier på tvers av tjenester:

| Mulighet | Beskrivelse |
|---|---|
| **Tverr-rammeverks interoperabilitet** | En agent bygget med ett rammeverk kan delegere oppgaver til en agent bygget med et hvilket som helst annet A2A-kompatibelt rammeverk, noe som muliggjør ekte interoperabilitet på tvers av organisasjoner. |
| **Tjenesteavgrensninger** | Agenter kan befinne seg i separate mikrotjenester, skyregioner, eller til og med forskjellige organisasjoner samtidig som de samarbeider sømløst. |
| **Dynamisk oppdagelse** | En orkestrator kan foreta spørringer mot et Agent Card-register ved kjøretid for å finne den mest egnede spesialisten for en gitt deloppgave. |
| **Streaming & push-varsler** | A2A støtter Server-Sent Events (SSE) for sanntidsoppdateringer om fremdrift og push-varsler for langvarige oppgaver. |

Arbeidsflyten vi bygde ovenfor er en forenklet versjon av dette mønsteret som kjører i samme prosess. I en ekte
implementasjon ville hver agent eksponere et HTTP-endepunkt, publisere en Agent Card, og kommunisere
via A2A JSON-RPC-protokollen.


## Sammendrag

I denne leksjonen lærte du:

1. **Hva A2A-protokollen er** — en åpen standard for agent-til-agent oppdagelse, meldingsutveksling,
   og oppgavehåndtering.
2. **Hvordan lage spesialiserte agenter** — en valutavekslingsagent, en aktivitetsplanleggeragent,
   og en Travel Manager-orkestrator.
3. **Hvordan koble agenter inn i en arbeidsflyt** — ved å bruke `WorkflowBuilder` for å modellere sekvensiell
   meldingsutveksling mellom agenter.
4. **Hvordan A2A fungerer i produksjon** — muliggjør samarbeid på tvers av rammeverk og tjenester
   med dynamisk oppdagelse og strømmende oppdateringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfraskrivelse:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet i sitt originalspråk bør anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
